## Section 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Section 2: Load and Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('./data/cleaned_comment_yt.csv')

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nColumn names:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDataset Info:")
print(df.info())

Dataset shape: (21945, 2)

First few rows:
                                            comments  \
0  Makin lama makin ke sini hukum semakin ga jela...   
1        Gila bnr padahal di pilih dari suara rakyat   
2  HOUWWWWW...AKAN. ANYAK RAKYAT YANG DI PENJARA....   
3    Seru bisa rakyat semua  turun kejal ini  mantap   
4  Kaya anak teka undang2..begituh ga berbobot bp...   

                                    cleaned_comments  
0  hukum jelasuu ampas aset tele tir hidup indonesia  
1                        gila bnr pilih suara rakyat  
2  hou anyak rakyat penjaradan siksa azab preside...  
3                     seru rakyat turun kejal mantap  
4  kaya anak teka undangbegituh bobot bpk undangy...  

Column names: ['comments', 'cleaned_comments']

Data types:
 comments            object
cleaned_comments    object
dtype: object

Missing values:
 comments            0
cleaned_comments    0
dtype: int64

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21945 entries, 0 to 

## Section 3: Preprocess Text Data

In [ ]:
# Use the cleaned_comments column for training
X = df['cleaned_comments'].values

# Handle missing values
X = np.array([str(x) if pd.notna(x) else "" for x in X])

print(f"Total samples: {len(X)}")
print(f"\nSample comments:")
for i in range(min(5, len(X))):
    print(f"{i+1}. {X[i][:100]}...")

# Load pre-trained Indonesian sentiment model to label the data
print("\n" + "="*80)
print("Loading pre-trained sentiment model to label your data...")
print("="*80)

from transformers import pipeline

# Use IndoBERT sentiment model for Indonesian text
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="mdhugol/indonesia-bert-sentiment-classification",
    device=0 if torch.cuda.is_available() else -1
)

# Analyze sentiment for all comments (in batches to avoid memory issues)
print("\nAnalyzing sentiment for all comments...")
batch_size = 32
sentiments = []

for i in range(0, len(X), batch_size):
    batch = X[i:i+batch_size].tolist()
    results = sentiment_analyzer(batch, truncation=True, max_length=512)
    sentiments.extend([r['label'] for r in results])
    if (i + batch_size) % 1000 == 0:
        print(f"Processed {min(i + batch_size, len(X))}/{len(X)} comments...")

print(f"Completed! Total processed: {len(sentiments)}")

# Map labels to numeric values
# The model 'mdhugol/indonesia-bert-sentiment-classification' typically outputs:
# 'LABEL_0': positive, 'LABEL_1': neutral, 'LABEL_2': negative
label_mapping = {
    'LABEL_0': 2,  # positive
    'LABEL_1': 1,  # neutral
    'LABEL_2': 0   # negative
}

# Map the raw string labels to numeric values using the corrected mapping
y = np.array([label_mapping.get(s, 1) for s in sentiments]) # Use s directly, not s.lower()

print(f"\nSentiment Label distribution:")
unique, counts = np.unique(y, return_counts=True)
label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
for label, count in zip(unique, counts):
    print(f"{label_names.get(label, label)}: {count} samples ({count/len(y)*100:.2f}%)")

# Save labeled dataset
labeled_df = df.copy()
labeled_df['sentiment'] = sentiments
labeled_df['sentiment_label'] = y
labeled_df.to_csv('./data/labeled_comment_yt.csv', index=False)
print(f"\nLabeled dataset saved to: ./data/labeled_comment_yt.csv")

Total samples: 21945

Sample comments:
1. hukum jelasuu ampas aset tele tir hidup indonesia...
2. gila bnr pilih suara rakyat...
3. hou anyak rakyat penjaradan siksa azab presiden wakil untyk azab dunia akhirat penjarakaaaan rakyatm...
4. seru rakyat turun kejal mantap...
5. kaya anak teka undangbegituh bobot bpk undangyg bagus makan uang rakyat mwkan uang pajakmakan duit k...

Loading pre-trained sentiment model to label your data...


Device set to use cuda:0



Analyzing sentiment for all comments...
Processed 4000/21945 comments...
Processed 8000/21945 comments...
Processed 12000/21945 comments...
Processed 16000/21945 comments...
Processed 20000/21945 comments...
Completed! Total processed: 21945

Sentiment Label distribution:
Negative: 14958 samples (68.16%)
Neutral: 3162 samples (14.41%)
Positive: 3825 samples (17.43%)

Labeled dataset saved to: ./data/labeled_comment_yt.csv


## Section 4: Tokenize Data with IndoBERT

In [ ]:
# Load IndoBERT tokenizer
tokenizer_name = "indolem/indobert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

print("Tokenizer loaded successfully!")
print(f"Tokenizer: {tokenizer_name}")

# Tokenize sample
sample_text = X[0]
encoded = tokenizer(sample_text, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
print(f"\nSample tokenization:")
print(f"Input IDs shape: {encoded['input_ids'].shape}")
print(f"Attention mask shape: {encoded['attention_mask'].shape}")

Tokenizer loaded successfully!
Tokenizer: indolem/indobert-base-uncased

Sample tokenization:
Input IDs shape: torch.Size([1, 128])
Attention mask shape: torch.Size([1, 128])


## Section 5: Prepare Dataset for Training

In [ ]:
# Split the data into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

# Create Custom Dataset class
class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label)
        }

# Create datasets
train_dataset = CommentDataset(X_train, y_train, tokenizer)
val_dataset = CommentDataset(X_val, y_val, tokenizer)
test_dataset = CommentDataset(X_test, y_test, tokenizer)

# Create dataloaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"\nDataLoaders created with batch size: {batch_size}")
print(f"Number of batches in training: {len(train_loader)}")

Training set size: 15361
Validation set size: 3292
Test set size: 3292

DataLoaders created with batch size: 16
Number of batches in training: 961


## Section 6: Configure and Initialize IndoBERT Model

In [ ]:
# Load pretrained IndoBERT model
model_name = "indolem/indobert-base-uncased"
num_labels = len(np.unique(y))  # Automatically detect number of sentiment classes
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
model.to(device)

print(f"Model loaded: {model_name}")
print(f"Model moved to device: {device}")
print(f"Number of labels: {num_labels}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")

# Training configuration
epochs = 3
learning_rate = 2e-5
num_training_steps = len(train_loader) * epochs
num_warmup_steps = int(0.1 * num_training_steps)

# Setup optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=learning_rate)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

print(f"\nTraining Configuration:")
print(f"Epochs: {epochs}")
print(f"Learning Rate: {learning_rate}")
print(f"Total Training Steps: {num_training_steps}")
print(f"Warmup Steps: {num_warmup_steps}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: indolem/indobert-base-uncased
Model moved to device: cuda
Number of labels: 3
Number of parameters: 110560515

Training Configuration:
Epochs: 3
Learning Rate: 2e-05
Total Training Steps: 2883
Warmup Steps: 288


## Section 7: Train the Model

In [ ]:
# Training loop
def train_epoch(model, train_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc="Training"):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_loader)
    return avg_loss

# Validation loop
def validate(model, val_loader, device):
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            total_loss += loss.item()

            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(preds)
            true_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(val_loader)
    accuracy = accuracy_score(true_labels, predictions)

    return avg_loss, accuracy, predictions, true_labels

# Train the model
print("Starting training...\n")
train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")

    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    train_losses.append(train_loss)
    print(f"Training Loss: {train_loss:.4f}")

    val_loss, val_accuracy, _, _ = validate(model, val_loader, device)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}\n")

print("Training completed!")

Starting training...

Epoch 1/3


Training: 100%|██████████| 961/961 [05:31<00:00,  2.90it/s]


Training Loss: 0.6934


Validating: 100%|██████████| 206/206 [00:23<00:00,  8.60it/s]


Validation Loss: 0.5524
Validation Accuracy: 0.7685

Epoch 2/3


...

Validation Loss: 0.4577
Validation Accuracy: 0.8256

Epoch 3/3


Training: 100%|██████████| 961/961 [05:30<00:00,  2.91it/s]


Training Loss: 0.3632


Validating: 100%|██████████| 206/206 [00:24<00:00,  8.58it/s]

Validation Loss: 0.4592
Validation Accuracy: 0.8290

Training completed!


## Section 8: Evaluate Model Performance

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_loss, test_accuracy, test_predictions, test_true_labels = validate(model, test_loader, device)

print(f"\nTest Results:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Calculate additional metrics
precision = precision_score(test_true_labels, test_predictions, average='weighted', zero_division=0)
recall = recall_score(test_true_labels, test_predictions, average='weighted', zero_division=0)
f1 = f1_score(test_true_labels, test_predictions, average='weighted', zero_division=0)

print(f"\nClassification Metrics:")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Create label names based on number of classes
label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
target_names = [label_names.get(i, f'Class {i}') for i in sorted(np.unique(test_true_labels))]

print(f"\nClassification Report:")
print(classification_report(test_true_labels, test_predictions, target_names=target_names))

print(f"\nConfusion Matrix:")
print(confusion_matrix(test_true_labels, test_predictions))

Evaluating on test set...


Validating: 100%|██████████| 206/206 [00:23<00:00,  8.61it/s]


Test Results:
Test Loss: 0.4541
Test Accuracy: 0.8357

Classification Metrics:
Precision: 0.8295
Recall: 0.8357
F1-Score: 0.8297

Classification Report:
              precision    recall  f1-score   support

    Negative       0.87      0.93      0.90      2244
     Neutral       0.74      0.54      0.63       475
    Positive       0.74      0.71      0.72       573

    accuracy                           0.84      3292
   macro avg       0.78      0.73      0.75      3292
weighted avg       0.83      0.84      0.83      3292


Confusion Matrix:
[[2088   63   93]
 [ 170  258   47]
 [ 140   28  405]]


## Section 9: Save the Trained Model

In [ ]:
# Save the model and tokenizer
model_save_path = './model/indobert_model'
tokenizer_save_path = './model/indobert_tokenizer'

# Create model directory if it doesn't exist
import os
os.makedirs('./model', exist_ok=True)

# Save model
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(tokenizer_save_path)

print(f"Model saved to: {model_save_path}")
print(f"Tokenizer saved to: {tokenizer_save_path}")

# Save training metrics
import json

metrics = {
    'test_loss': float(test_loss),
    'test_accuracy': float(test_accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'val_accuracies': val_accuracies
}

metrics_path = './model/indobert_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=4)

print(f"Metrics saved to: {metrics_path}")

print("\nTraining pipeline completed successfully!")

Model saved to: ./model/indobert_model
Tokenizer saved to: ./model/indobert_tokenizer
Metrics saved to: ./model/indobert_metrics.json

Training pipeline completed successfully!
